In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import time
import gc

# Memory management
tf.config.experimental.enable_memory_growth = True
tf.random.set_seed(42)

def clear_memory():
    """Clear GPU and system memory"""
    tf.keras.backend.clear_session()
    gc.collect()
    if tf.config.list_physical_devices('GPU'):
        tf.config.experimental.reset_memory_stats('GPU:0')

def prepare_data():
    """Load CIFAR-100 data (first 20 classes only)"""
    print("Loading CIFAR-100...")
    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data()

    # Select first 20 classes
    train_mask = y_train.flatten() < 20
    test_mask = y_test.flatten() < 20

    x_train = x_train[train_mask][:4000]  # Limit training samples
    y_train = y_train[train_mask][:4000]
    x_test = x_test[test_mask][:1000]     # Limit test samples
    y_test = y_test[test_mask][:1000]

    # Preprocess
    x_train = tf.cast(x_train, tf.float32) / 255.0
    x_test = tf.cast(x_test, tf.float32) / 255.0
    y_train = keras.utils.to_categorical(y_train, 20)
    y_test = keras.utils.to_categorical(y_test, 20)

    print(f"Train: {len(x_train)}, Test: {len(x_test)}")
    return (x_train, y_train), (x_test, y_test)

def train_single_model(model_name, model_class, train_data, test_data):
    """Train and evaluate one model, then clean up memory"""
    print(f"\nTraining {model_name}...")
    clear_memory()

    x_train, y_train = train_data
    x_test, y_test = test_data

    # Handle different input sizes
    if model_name in ['InceptionV3', 'Xception', 'InceptionResNetV2']:
        input_shape = (75, 75, 3)
        x_train_resized = tf.image.resize(x_train, [75, 75])
        x_test_resized = tf.image.resize(x_test, [75, 75])
    else:
        input_shape = (64, 64, 3)
        x_train_resized = tf.image.resize(x_train, [64, 64])
        x_test_resized = tf.image.resize(x_test, [64, 64])

    try:
        # Create base model
        base_model = model_class(
            weights='imagenet',
            include_top=False,
            input_shape=input_shape
        )
        base_model.trainable = False

        # Create full model
        model = keras.Sequential([
            base_model,
            keras.layers.GlobalAveragePooling2D(),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dropout(0.5),
            keras.layers.Dense(20, activation='softmax')
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # Train (shorter epochs to save time/memory)
        start_time = time.time()

        model.fit(
            x_train_resized, y_train,
            batch_size=16,  # Smaller batch size
            epochs=3,       # Fewer epochs
            validation_data=(x_test_resized, y_test),
            verbose=0
        )

        training_time = time.time() - start_time

        # Evaluate
        loss, accuracy = model.evaluate(x_test_resized, y_test, verbose=0)
        params = model.count_params()

        result = {
            'accuracy': accuracy,
            'loss': loss,
            'time': training_time,
            'params': params
        }

        print(f"{model_name}: Acc={accuracy:.4f}, Time={training_time:.1f}s")

        # Clean up
        del model, base_model
        clear_memory()

        return result

    except Exception as e:
        print(f"Error with {model_name}: {e}")
        clear_memory()
        return None

def main():
    """Main comparison function"""

    # Load data once
    train_data, test_data = prepare_data()

    # Model list (start with lighter models)
    models = [
        ('MobileNetV2', keras.applications.MobileNetV2),
        ('EfficientNetB0', keras.applications.EfficientNetB0),
        ('VGG16', keras.applications.VGG16),
        ('ResNet50', keras.applications.ResNet50),
        ('DenseNet121', keras.applications.DenseNet121),
        ('VGG19', keras.applications.VGG19),
        ('InceptionV3', keras.applications.InceptionV3),
        ('ResNet101', keras.applications.ResNet101),
        ('Xception', keras.applications.Xception),
        ('InceptionResNetV2', keras.applications.InceptionResNetV2),
    ]

    results = {}

    # Train models one by one
    for model_name, model_class in models:
        result = train_single_model(model_name, model_class, train_data, test_data)
        if result:
            results[model_name] = result

    # Display results
    if results:
        df = pd.DataFrame(results).T
        df = df.sort_values('accuracy', ascending=False)

        print("\n" + "="*60)
        print("FINAL RESULTS")
        print("="*60)
        print(df.round(4))
        print(f"\nBest Model: {df.index[0]} ({df.iloc[0]['accuracy']:.4f} accuracy)")

        return df
    else:
        print("No models completed successfully!")
        return None

if __name__ == "__main__":
    results = main()

    print("\nComparison completed!")

Starting memory-efficient CNN comparison...
Using reduced dataset and image sizes to prevent crashes.
Loading CIFAR-100...
Train: 4000, Test: 1000

Training MobileNetV2...


/tmp/ipython-input-1-58642605.py:62: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = model_class(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MobileNetV2: Acc=0.5680, Time=22.2s

Training EfficientNetB0...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EfficientNetB0: Acc=0.0450, Time=36.5s

Training VGG16...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
VGG16: Acc=0.4870, Time=15.9s

Training ResNet50...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet50: Acc=0.0630, Time=37.9s

Training DenseNet121...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
DenseNet121: Acc=0.5630, Time=63.3s

Training VGG19...
80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
VGG19: Acc=0.4070, Time=21.0s

Training InceptionV3...
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
InceptionV3: Acc=0.5150, Time=35.6s

Training ResNet101...
171446536/171446536 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
ResNet101: Acc=0.0520, Time=74.0s

Training Xception...
83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Xception: Acc=0.6520, Time=26.3s

Training InceptionResNetV2...
219055592/219055592 